In [18]:
# mp3 -> wav 增量更新

import os
import subprocess
import time

MP3_DIR = "outputs/mp3"
WAV_DIR = "outputs/wav"

def batch_mp3_to_wav(mp3_dir: str, wav_dir: str, sample_rate: str = "16000"):
    if not os.path.isdir(mp3_dir):
        raise FileNotFoundError(f"❌ 输入目录不存在：{mp3_dir}")

    os.makedirs(wav_dir, exist_ok=True)

    # 以文件名（不含后缀）做差集
    mp3_files = {
        f[:-4] for f in os.listdir(mp3_dir)
        if f.lower().endswith(".mp3")
        and os.path.isfile(os.path.join(mp3_dir, f))
    }
    wav_files = {
        f[:-4] for f in os.listdir(wav_dir)
        if f.lower().endswith(".wav")
        and os.path.isfile(os.path.join(wav_dir, f))
    }

    to_convert = sorted(mp3_files - wav_files)
    skipped = len(mp3_files & wav_files)

    if skipped:
        print(f"⏭️ 已存在跳过：{skipped} 个")
    if not to_convert:
        print("✅ 全部已同步，无需转换")
        return

    print(f"🎬 待转换 {len(to_convert)} 个\n")

    total = len(to_convert)
    t_start = time.time()

    for i, name in enumerate(to_convert, 1):
        mp3_path = os.path.join(mp3_dir, f"{name}.mp3")
        wav_path = os.path.join(wav_dir, f"{name}.wav")

        cmd = [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", mp3_path,
            "-ar", sample_rate,
            "-ac", "1",
            wav_path
        ]

        s = time.time()
        subprocess.run(cmd, check=True)
        cost = time.time() - s

        print(f"[{i}/{total}] ✅ {name}.wav  ({cost:.2f}s)")

    print(f"\n🏁 完成！共转换 {total} 个，总耗时 {time.time() - t_start:.2f}s")


if __name__ == "__main__":
    batch_mp3_to_wav(MP3_DIR, WAV_DIR, sample_rate="16000")

⏭️ 已存在跳过：1 个
🎬 待转换 3 个



python(25782) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[1/3] ✅ 2K_4_en.wav  (0.61s)


python(25783) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[2/3] ✅ Mt_10_en.wav  (0.29s)


python(25784) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


[3/3] ✅ Rom_6_en.wav  (0.26s)

🏁 完成！共转换 3 个，总耗时 1.16s


In [9]:
import os
import subprocess

# 找到 aligner 环境路径
conda_prefix = subprocess.check_output(
    ["conda", "info", "--base"], text=True
).strip()

aligner_bin = os.path.join(conda_prefix, "envs", "aligner", "bin")
os.environ["PATH"] = aligner_bin + ":" + os.environ["PATH"]

# 验证
!mfa version

python(84858) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(84859) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


3.3.9


In [19]:
import os
import shutil

WAV_DIR = "outputs/wav"
TXT_DIR = "outputs/plaintext"
CORPUS_DIR = "corpus"

def batch_prepare_corpus(wav_dir: str, txt_dir: str, corpus_dir: str):
    if not os.path.isdir(wav_dir):
        raise FileNotFoundError(f"❌ wav 目录不存在：{wav_dir}")
    if not os.path.isdir(txt_dir):
        raise FileNotFoundError(f"❌ plaintext 目录不存在：{txt_dir}")

    os.makedirs(corpus_dir, exist_ok=True)

    # 以文件名（不含后缀）做差集
    wav_files = {
        f[:-4] for f in os.listdir(wav_dir)
        if f.lower().endswith(".wav")
        and os.path.isfile(os.path.join(wav_dir, f))
    }
    txt_files = {
        f[:-4] for f in os.listdir(txt_dir)
        if f.lower().endswith(".txt")
        and os.path.isfile(os.path.join(txt_dir, f))
    }
    corpus_files = {
        f[:-4] for f in os.listdir(corpus_dir)
        if os.path.isfile(os.path.join(corpus_dir, f))
    }

    # 需要同时有 wav 和 txt 才能入库
    ready = wav_files & txt_files
    to_copy = sorted(ready - corpus_files)
    skipped = len(ready & corpus_files)

    if skipped:
        print(f"⏭️ 已存在跳过：{skipped} 个")
    if not to_copy:
        print("✅ corpus 已全部同步，无需复制")
        return

    print(f"📦 待复制到 corpus：{len(to_copy)} 个\n")

    for i, name in enumerate(to_copy, 1):
        shutil.copy(
            os.path.join(wav_dir, f"{name}.wav"),
            os.path.join(corpus_dir, f"{name}.wav")
        )
        shutil.copy(
            os.path.join(txt_dir, f"{name}.txt"),
            os.path.join(corpus_dir, f"{name}.txt")
        )
        print(f"[{i}/{len(to_copy)}] ✅ {name}")

    print(f"\n🏁 corpus 准备完成，共新增 {len(to_copy)} 个")


if __name__ == "__main__":
    batch_prepare_corpus(WAV_DIR, TXT_DIR, CORPUS_DIR)

⏭️ 已存在跳过：1 个
📦 待复制到 corpus：3 个

[1/3] ✅ 2K_4_en
[2/3] ✅ Mt_10_en
[3/3] ✅ Rom_6_en

🏁 corpus 准备完成，共新增 3 个


In [21]:
import os
import shutil
import subprocess

# ===== 目录 =====
WAV_DIR = "outputs/wav"
TXT_DIR = "outputs/plaintext"
CORPUS_DIR = "corpus"
ALIGN_OUT = "forcealign"

# ===== MFA 模型路径（✅ 关键修复）=====
dict_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/dictionary/english_us_arpa.dict"
)
acoustic_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/acoustic/english_us_arpa.zip"
)

# ===== 注入 aligner 环境 =====
conda_prefix = subprocess.check_output(
    ["conda", "info", "--base"], text=True
).strip()
aligner_bin = os.path.join(conda_prefix, "envs", "aligner", "bin")
os.environ["PATH"] = aligner_bin + ":" + os.environ["PATH"]

def prepare_and_align():
    os.makedirs(CORPUS_DIR, exist_ok=True)
    os.makedirs(ALIGN_OUT, exist_ok=True)

    # 已对齐结果
    aligned = {
        f[:-9] for f in os.listdir(ALIGN_OUT)
        if f.endswith(".TextGrid")
    }

    wavs = {f[:-4] for f in os.listdir(WAV_DIR) if f.endswith(".wav")}
    txts = {f[:-4] for f in os.listdir(TXT_DIR) if f.endswith(".txt")}

    ready = wavs & txts
    to_align = sorted(ready - aligned)

    if not to_align:
        print("✅ 所有样本已完成强制对齐")
        return

    print(f"🎯 待对齐：{len(to_align)} 个")

    for name in to_align:
        shutil.copy(
            os.path.join(WAV_DIR, f"{name}.wav"),
            os.path.join(CORPUS_DIR, f"{name}.wav")
        )
        shutil.copy(
            os.path.join(TXT_DIR, f"{name}.txt"),
            os.path.join(CORPUS_DIR, f"{name}.txt")
        )

    cmd = [
        "mfa", "align", CORPUS_DIR,
        dict_path,
        acoustic_path,
        ALIGN_OUT,
        "--clean", "--overwrite"
    ]

    print("🚀 开始强制对齐...\n")
    subprocess.run(cmd, check=True)
    print("\n🏁 强制对齐完成")

if __name__ == "__main__":
    prepare_and_align()

python(25902) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


🎯 待对齐：3 个
🚀 开始强制对齐...



python(25904) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   


   4% ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/100  [ 0:00:01 < -:--:-- , ? it/s ]


 INFO     Found 1 speaker across 4 files, average number of utterances per      
          speaker: 4.0                                                          
 INFO     Initializing multiprocessing jobs...                                  
 WARNING  Number of jobs was specified as 3, but due to only having 1 speakers, 
          MFA will only use 1 jobs. Use the --single_speaker flag if you would  
          like to split utterances across jobs regardless of their speaker.     
 INFO     Normalizing text...                                                   


...

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:02 < 0:00:00 , ? it/s ]


 INFO     Analyzing alignment quality...                                        


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , ? it/s ]
   0% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/4  [ 0:00:00 < -:--:-- , ? it/s ]

 INFO     Exporting alignment TextGrids to forcealign...                        
 INFO     Finished exporting TextGrids to forcealign!                           
 INFO     Done! Everything took 66.197 seconds                                  


 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:00 < 0:00:00 , ? it/s ]

🏁 强制对齐完成


In [1]:
# 连接数据库

import sqlite3

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

In [8]:
# 清空数据库时间戳（要先连接数据库）

cursor.execute("""
UPDATE words
SET start_time = NULL,
    end_time = NULL;
""")
conn.commit()

In [17]:
# 把强制对齐生成的时间戳 TextGrid 导入数据库，毫秒级整数

import sqlite3
import re

def parse_textgrid(textgrid_path):
    intervals = []
    current = None
    in_words_tier = False

    with open(textgrid_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith('name = "words"'):
                in_words_tier = True
                continue
            if line.startswith('item ['):
                in_words_tier = False
                continue

            if not in_words_tier:
                continue

            # ✅ 区间开始
            if re.match(r'intervals\s*\[\d+\]:', line):
                current = {'xmin': None, 'xmax': None, 'text': ''}
                continue

            # ✅ 关键修复：current 不存在就跳过
            if current is None:
                continue

            if m := re.match(r'xmin\s*=\s*([\d\.]+)', line):
                current['xmin'] = float(m.group(1))
            elif m := re.match(r'xmax\s*=\s*([\d\.]+)', line):
                current['xmax'] = float(m.group(1))
            elif m := re.match(r'text\s*=\s*"([^"]*)"', line):
                word = m.group(1).strip()
                if word and current['xmin'] is not None:
                    intervals.append((
                        int(round(current['xmin'] * 1000)),
                        int(round(current['xmax'] * 1000)),
                        word
                    ))
                current = None  # ✅ 强制重置

    return intervals


def update_words_timestamps(db_path, textgrid_path):
    intervals = parse_textgrid(textgrid_path)
    if not intervals:
        print("❌ 未提取到单词时间戳")
        return

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        cursor.execute("SELECT id, word FROM words WHERE type = 'word' ORDER BY id")
        records = cursor.fetchall()

        idx = 0
        for word_id, word_text in records:
            if idx >= len(intervals):
                print("⚠️ TextGrid 单词不足")
                break

            start_ms, end_ms, tg_word = intervals[idx]

            if word_text.lower() != tg_word.lower():
                print(f"⚠️ 不匹配：DB={word_text} ≠ TG={tg_word}")
                idx += 1
                continue

            cursor.execute("""
                UPDATE words
                SET start_time = ?, end_time = ?
                WHERE id = ?
            """, (start_ms, end_ms, word_id))

            idx += 1

        conn.commit()
        print(f"✅ 成功更新 {idx} 个单词的时间戳（毫秒）")

    except Exception as e:
        print("❌ 更新失败：", e)
        conn.rollback()
    finally:
        conn.close()


# ------------------- 调用 -------------------
db_path = "db/bible.db"
textgrid_path = "forcealign/Mt_1_en.TextGrid"

update_words_timestamps(db_path, textgrid_path)

✅ 成功更新 533 个单词的时间戳（毫秒）


In [31]:
# drop列操作需要查看版本

import sqlite3
sqlite3.sqlite_version

'3.51.2'

In [32]:
# drop列

import sqlite3

conn = sqlite3.connect("db/bible.db")
cur = conn.cursor()

cur.execute("ALTER TABLE words DROP COLUMN start_time")
cur.execute("ALTER TABLE words DROP COLUMN end_time")
conn.commit()

In [33]:
# 添加时间戳列为整数类型

cur.execute("ALTER TABLE words ADD COLUMN start_time INTEGER")
cur.execute("ALTER TABLE words ADD COLUMN end_time INTEGER")
conn.commit()

In [39]:
# 导出数据库人名

!sqlite3 db/bible.db \
  "SELECT DISTINCT UPPER(name_en) FROM persons WHERE name_en IS NOT NULL;" \
  > db/name_en.txt

python(13027) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [43]:
# 生成人名词典

!mfa g2p db/name_en.txt english_us_arpa forcealign/names_en.dict

python(13302) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


 INFO     Generating pronunciations...                                          
   0% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/0  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Done! Everything took 8.656 seconds                                   


In [5]:
# 合并词典

!cat ~/Documents/MFA/pretrained_models/dictionary/english_mfa.dict \
     forcealign/names_en.dict \
     > forcealign/english_add_bible_names.dict

In [7]:
with open("forcealign/names_en.dict", "w", encoding="utf-8") as f:
    f.write("JEHOSHAPHAT  JH AH HH OW SH AH F AE T\n")
    f.write("JECHONIAH    JH IH K OW N AY AH\n")

In [39]:
# =========================
# TextGrid → timestamps
# ✅ 只处理 item [1]（words tier）
# =========================

import re
import sqlite3
import os

DB_PATH = "db/bible.db"
TEXTGRID_DIR = "outputs/force_align"

# ========= 交互输入 =========
book_abbr = input("请输入书卷简称（如 2K）：").strip()
chapter = int(input("请输入章数（如 4）：").strip())

# =========================

filename = f"{book_abbr}_{chapter}_en.TextGrid"
filepath = os.path.join(TEXTGRID_DIR, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(f"❌ 找不到文件: {filepath}")

with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()

# ======================
# 1️⃣ 只提取 item [1] 区块
# ======================
item1_pattern = re.compile(
    r"item\s*\[\]:\s*\n\s*item\s*\[1\]:(.*?)(?=\n\s*item\s*\[\]:\s*\n\s*item\s*\[2\]:|\Z)",
    re.DOTALL
)

m = item1_pattern.search(content)
if not m:
    raise ValueError("❌ 未找到 item [1]，请检查 TextGrid 结构")

item1_content = m.group(1)

# ======================
# 2️⃣ 解析 intervals
# ======================
interval_pattern = re.compile(
    r"intervals\s*\[\d+\]:\s*\n"
    r"\s*xmin\s*=\s*([\d.]+)\s*\n"
    r"\s*xmax\s*=\s*([\d.]+)\s*\n"
    r'\s*text\s*=\s*"([^"]*)"',
    re.MULTILINE
)

matches = interval_pattern.findall(item1_content)

# ======================
# 3️⃣ 写入数据库
# ======================
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='timestamps'"
)
if cursor.fetchone() is None:
    raise RuntimeError("❌ timestamps 表不存在，请先建表")

word_seq = 0
for xmin, xmax, text in matches:
    if text.strip() == "":
        continue

    word_seq += 1

    start_ms = int(round(float(xmin) * 1000, 0))
    end_ms = int(round(float(xmax) * 1000, 0))

    row_id = f"{book_abbr}.{chapter}.{word_seq}"

    cursor.execute(
        """
        INSERT INTO timestamps (
            id, book_abbr, chapter, word_seq, word, start_time, end_time
        ) VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (row_id, book_abbr, chapter, word_seq, text, start_ms, end_ms)
    )

    print(f"{row_id:15s} | {text:10s} | {start_ms:>6} | {end_ms:>6}")

conn.commit()
conn.close()

print(f"\n✅ 共插入 {word_seq} 条记录（仅 item [1]，已忽略空 text）")

请输入书卷简称（如 2K）：Mt
请输入章数（如 4）：1
Mt.1.1          | an         |    130 |    250
Mt.1.2          | account    |    250 |    740
Mt.1.3          | of         |    740 |    850
Mt.1.4          | the        |    850 |    950
Mt.1.5          | genealogy  |    950 |   1830
Mt.1.6          | of         |   1830 |   1950
Mt.1.7          | jesus      |   1950 |   2450
Mt.1.8          | the        |   2450 |   2580
Mt.1.9          | messiah    |   2580 |   3280
Mt.1.10         | the        |   3550 |   3680
Mt.1.11         | son        |   3680 |   3940
Mt.1.12         | of         |   3940 |   4090
Mt.1.13         | david      |   4090 |   4580
Mt.1.14         | the        |   4840 |   4940
Mt.1.15         | son        |   4940 |   5210
Mt.1.16         | of         |   5210 |   5350
Mt.1.17         | abraham    |   5350 |   6080
Mt.1.18         | abraham    |   6690 |   7280
Mt.1.19         | was        |   7280 |   7440
Mt.1.20         | the        |   7440 |   7530
Mt.1.21         | father     |